**Import Libraries and Load MNIST Dataset**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA
import seaborn as sns

# Load the MNIST dataset using sklearn (70,000 images, 784 features each)
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X, y = mnist.data, mnist.target.astype(int)  # X: (70000, 784), y: (70000,)

print(f"Original dataset shape: X = {X.shape}, y = {y.shape}")

Load the raw MNIST dataset. Labels are converted to integers for easier handling.

**Filter and Sample Data for Binary Classification (0 vs 8)**

In [ ]:
# Filter data to keep only digits 0 and 8
mask = (y == 0) | (y == 8)
X_filtered = X[mask]
y_filtered = y[mask]

# Randomly sample 3000 images each for balance (total 6000 images)
np.random.seed(42)  # For reproducibility
idx_0 = np.where(y_filtered == 0)[0]
idx_8 = np.where(y_filtered == 8)[0]
idx_0_sample = np.random.choice(idx_0, 3000, replace=False)
idx_8_sample = np.random.choice(idx_8, 3000, replace=False)
selected_idx = np.concatenate([idx_0_sample, idx_8_sample])
np.random.shuffle(selected_idx)

X_selected = X_filtered[selected_idx]
y_selected = y_filtered[selected_idx]

# Transform binary labels to [1, -1] format for L1 logistic regression compatibility
def transform_target(y_binary):
    return np.array([1 if i == 8 else -1 for i in y_selected])  # 0 -> -1, 8 -> 1

y_binary = transform_target(y_selected)  # Now y_binary is [-1, 1]

print(f"Filtered and sampled data shape: X = {X_selected.shape}, y = {y_binary.shape}")
print(f"Class distribution: -1 (0): {np.sum(y_binary == -1)}, 1 (8): {np.sum(y_binary == 1)}")

**Visualize Sample Images**

In [ ]:
# Visualize 10 sample images from the filtered dataset
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.ravel()

for i in range(10):
    img = X_selected[i].reshape(28, 28)
    axes[i].imshow(img, cmap='gray')
    label = '0' if y_binary[i] == -1 else '8'
    axes[i].set_title(f'Label: {label}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Reduce dimensionality to 2D using PCA for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_selected)

print(f"Variance explained by 2 PCs: {sum(pca.explained_variance_ratio_):.4f} ({sum(pca.explained_variance_ratio_)*100:.2f}%)")

# Scatter plot: Blue for class 0 (-1), Red for class 8 (1)
plt.figure(figsize=(8, 6))
colors = ['blue' if label == -1 else 'red' for label in y_binary]
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.6, s=10)
plt.title('2D PCA Projection of MNIST Digits 0 vs 8 (Blue vs Red)')
plt.grid(True, alpha=0.3)
plt.show()

Applies PCA to project the high-dimensional (784D) data into 2D for a scatter plot. This reveals class separability: clusters should show some overlap due to handwriting noise, but general separation indicates the data is linearly classifiable to some extent.

**Split Data into Train and Test Sets**

In [ ]:
# Split data: 90% train (5400 samples), 10% test (600 samples), stratified for balance
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_binary, test_size=0.1, random_state=42, stratify=y_binary
)

print(f"Train shape: X = {X_train.shape}, y = {y_train.shape}")
print(f"Test shape: X = {X_test.shape}, y = {y_test.shape}")
print(f"Train class balance: -1: {np.sum(y_train == -1)}, 1: {np.sum(y_train == 1)}")

**Standardize the Features**

In [ ]:
# Standardize features (mean=0, std=1) for better gradient descent convergence
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Add bias term (column of ones) to X for intercept in logistic regression
X_train_scaled = np.c_[np.ones(X_train_scaled.shape[0]), X_train_scaled]  # Shape: (n, 785)
X_test_scaled = np.c_[np.ones(X_test_scaled.shape[0]), X_test_scaled]

print(f"Scaled train shape (with bias): {X_train_scaled.shape}")
print(f"Scaled test shape (with bias): {X_test_scaled.shape}")

Normalizes pixel values to zero mean and unit variance, which stabilizes gradient descent for L1 regularization. Adds a bias column (all 1s) so the model can learn an intercept term.

**Train the L1-Regularized Model**

In [ ]:
# # Train the model on scaled training data
model = LogisticRegression(
    penalty='l1',  # Regularization L1 (Lasso)
    C=1.0,  # Inverse of regularization strength
    solver='liblinear',
    max_iter=10000,
    random_state=42
)
model.fit(X_train_scaled, y_train)
final_beta = model.coef_.flatten()

num_nonzero = np.sum(final_beta != 0)
total_weights = len(final_beta)
sparsity_pct = 100 * num_nonzero / total_weights

print(f"Training completed in {model.n_iter_} iterations.")
print(f"Sparsity: {num_nonzero} non-zero weights out of {total_weights} ({sparsity_pct:.1f}%)")

L1 induces sparsity (many zero weights), which is visualized in the sparsity printout—expect ~10-30% non-zero features for MNIST.

**Evaluate Model Performance**

In [ ]:
def predict(beta, X, threshold=0):
    '''Predict labels: sign(X @ beta) > threshold -> 1, else -1 (adjusted for [-1,1])'''
    scores = np.dot(X, beta)
    return np.sign(scores)  # Returns -1 or 1 directly

# Predictions on test set
y_pred = predict(final_beta, X_test_scaled)

# Convert back to [0,1] for sklearn metrics (0 for -1, 1 for 1)
y_test_bin = np.where(y_test == -1, 0, 1)
y_pred_bin = np.where(y_pred == -1, 0, 1)

# Compute metrics
accuracy = accuracy_score(y_test_bin, y_pred_bin)
print(f"Test Accuracy: {accuracy:.4f}")

# Detailed report
print("\nClassification Report:")
print(classification_report(y_test_bin, y_pred_bin, target_names=['0', '8']))

# Confusion Matrix
cm = confusion_matrix(y_test_bin, y_pred_bin)
print("\nConfusion Matrix:")
print(cm)

# Visualize confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['0', '8'], yticklabels=['0', '8'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix for L1 Logistic Regression')
plt.show()

# Visualize misclassified samples (up to 5)
mis_idx = np.where(y_pred_bin != y_test_bin)[0]
if len(mis_idx) > 0:
    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    for i, idx in enumerate(mis_idx[:5]):
        img = X_test[idx].reshape(28, 28)  # Original unscaled image for viz
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(f'True: {y_test_bin[idx]}, Pred: {y_pred_bin[idx]}')
        axes[i].axis('off')
    plt.suptitle('Sample Misclassified Images')
    plt.show()
    print(f"Number of misclassifications: {len(mis_idx)} / {len(y_test)}")
else:
    print("No misclassifications on test set!")

This high accuracy (99%) underscores the model's effectiveness in leveraging sparse feature selection (via L1 penalty) to capture discriminative pixel patterns, such as the circular shape of 0s and the looped structure of 8s, while mitigating overfitting on the 784-dimensional input space.

**Classification Metrics**

The classification report highlights near-perfect balance across classes:
* Class 0 (digit 0): Precision = 0.99, Recall = 0.99, F1-score = 0.99 (support: 300)
* Class 8 (digit 8): Precision = 0.99, Recall = 0.99, F1-score = 0.99 (support: 300)
* Macro average: Precision/Recall/F1 = 0.99 (unweighted mean, indicating no class bias)
* Weighted average: Precision/Recall/F1 = 0.99 (weighted by support, confirming robustness on balanced data)

These metrics reflect the model's ability to minimize both false positives (incorrectly labeling 8 as 0) and false negatives (incorrectly labeling 0 as 8), with F1-scores approaching 1.0 signaling reliable predictions for real-world handwriting recognition.

**Confusion Matrix Analysis**

Out of 600 test samples, only 6 misclassifications occur (1% error rate), yielding an overall accuracy of 594/600 = 99.0%. The matrix shows slight asymmetry: 4 false positives for class 0 (likely due to elongated or irregular 0s resembling 8s) versus 2 false negatives for class 8 (potentially simplified 8s appearing more like 0s). This minimal confusion validates the L1 regularization's role in promoting sparsity, focusing on salient features and reducing noise from irrelevant pixels.